# Delta Lake — Advanced Practical Topics

Short recap of Delta Lake concepts from Fundamentals, then deep dive into Change Data Feed (CDF), advanced MERGE patterns, and practical maintenance.

| Training Block | Duration | Type |
|---|---|---|
| Delta Lake Advanced — Demo | 20 min | Demo |

**Prerequisites:** Databricks Fundamentals (Delta ACID, MERGE, Time Travel)

## Learning Objectives

After completing this module you will be able to:

- **Enable** and use Change Data Feed (CDF) for incremental ETL
- **Apply** advanced MERGE patterns (conditional updates, delete handling)
- **Use** RESTORE for disaster recovery (revert to previous version)
- **Compare** Shallow Clone vs Deep Clone for testing, migration, and backups
- **Understand** Delta Lake good practices for analytical workloads

## Setup

In [0]:
%run ../../setup/00_setup

### Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timedelta

# Display user context
display(
    spark.createDataFrame([
        (CATALOG, BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA)
    ], ['catalog', 'bronze_schema', 'silver_schema', 'gold_schema'])
)

# Set catalog and schema as default
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

## CRUD Operations & MERGE

INSERT, UPDATE, DELETE and MERGE INTO operations on Delta tables. MERGE is the most important in practice — it appears on almost every test.

**Theoretical Introduction:**

Delta Lake supports the full range of CRUD operations (Create, Read, Update, Delete), making it ideal for transactional workloads in Data Lake. All operations are:
- **Atomic**: Either fully complete or fully rolled back
- **ACID-compliant**: Ensuring data consistency
- **Recorded in Delta Log**: Full audit trail of all changes

Additionally, Delta Lake provides the powerful **MERGE INTO** operation (also known as "upsert") that combines INSERT and UPDATE in a single atomic transaction - essential for CDC (Change Data Capture) scenarios.

### Example: MERGE INTO (Upsert)

**Objective:** Demonstration of upsert operation - update existing and insert new records in a single atomic transaction

**MERGE INTO syntax:**

```sql
MERGE INTO target
USING source ON target.key = source.key
WHEN MATCHED THEN UPDATE SET ...
WHEN NOT MATCHED THEN INSERT (...) VALUES (...)
WHEN NOT MATCHED BY SOURCE THEN DELETE -- optional
```

| Clause | Description |
|---|---|
| `WHEN MATCHED` | Executed when a source row matches a target row (UPDATE or DELETE) |
| `WHEN NOT MATCHED` | Executed when the source row has no match in target (INSERT) |
| `WHEN NOT MATCHED BY SOURCE` | Executed when the target row has no match in source (DELETE) |
| `AND condition` | Optional extra filter on any `WHEN` clause |

MERGE INTO is especially useful when processing changes from transactional systems (CDC patterns). It allows you to:
- **Update** existing records when a match is found
- **Insert** new records when no match exists
- **Delete** records based on conditions (optional)

<img src="../../../assets/images/ff01677d3d4a45d6a6a7530d8911b785.png" width="800">

**MERGE INTO — Syntax Reference**

| Clause | Syntax |
|--------|--------|
| Match condition | `MERGE INTO target t USING source s ON t.key = s.key` |
| Update on match | `WHEN MATCHED THEN UPDATE SET *` |
| Conditional update | `WHEN MATCHED AND s.val <> t.val THEN UPDATE SET *` |
| Insert on no match | `WHEN NOT MATCHED THEN INSERT *` |
| Delete on match | `WHEN MATCHED AND s.op = 'D' THEN DELETE` |
| Python API | `DeltaTable.forName(spark, 't').merge(src, 't.id = s.id').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()` |

In [0]:
# Load customer data and create Delta table for MERGE demo
customers_df = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{DATASET_PATH}/customers/customers.csv")
)

spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.customers_delta")

In [0]:
customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta")

print(f"Table {CATALOG}.{BRONZE_SCHEMA}.customers_delta created with {customers_df.count()} rows")
display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta").limit(5))

In [0]:
# Prepare data for merge (mix of updates and new records)
merge_data = spark.createDataFrame([
    ("CUST010001", "Updated", "Name", "updated@example.com", "+48 999 999 999", "Poznan", "WP", "Poland", "2023-12-01", "VIP", "Diamond"),  # Update existing
    ("CUST030001", "Brand", "New", "brand.new@example.com", "+48 777 777 777", "Wroclaw", "DS", "Poland", "2023-12-15", "Basic", "Bronze"),   # Insert new
    ("CUST030002", "Another", "New", "another.new@example.com", "+48 888 888 888", "Lodz", "LD", "Poland", "2023-12-16", "Premium", "Silver") # Insert new
], ["customer_id", "first_name", "last_name", "email", "phone", "city", "state", "country", "registration_date", "customer_segment", "customer_tier"])

# Create temporary view for merge operation
merge_data.createOrReplaceTempView("customer_updates")

In [0]:
%sql
select * from customer_updates

In [0]:
# MERGE INTO operation (Upsert)
spark.sql(f"""
    MERGE INTO {CATALOG}.{BRONZE_SCHEMA}.customers_delta AS target
    USING customer_updates AS source
    ON target.customer_id = source.customer_id

    WHEN MATCHED THEN
        UPDATE SET
            target.first_name = source.first_name,
            target.last_name = source.last_name,
            target.email = source.email,
            target.phone = source.phone,
            target.city = source.city,
            target.state = source.state,
            target.country = source.country,
            target.registration_date = source.registration_date,
            target.customer_segment = source.customer_segment

    WHEN NOT MATCHED THEN
        INSERT (customer_id, first_name, last_name, email, phone, city, state, country, registration_date, customer_segment)
        VALUES (source.customer_id, source.first_name, source.last_name, source.email, source.phone, source.city, source.state, source.country, source.registration_date, source.customer_segment)
""")

In [0]:
# Verify MERGE results
display(
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta")
    .filter(F.col("customer_id").isin(["CUST010001", "CUST030001", "CUST030002"]))
    .orderBy("customer_id")
)

### DeltaTable Python API — Programmatic DML

The `delta.tables.DeltaTable` class provides a Python (non-SQL) interface for UPDATE, DELETE, and MERGE — useful when conditions or data come from Python variables.

```python
from delta.tables import DeltaTable

dt = DeltaTable.forName(spark, "catalog.schema.table")

# UPDATE
dt.update(condition="age < 18", set={"segment": "'junior'"})

# DELETE
dt.delete("status = 'inactive'")

# MERGE (upsert) — equivalent to SQL MERGE INTO
dt.alias("target").merge(
    source_df.alias("source"),
    "target.id = source.id"
) .whenMatchedUpdateAll() .whenNotMatchedInsertAll() .execute()
```

> **Pro Tip:** Both SQL `MERGE INTO` and `DeltaTable.merge()` produce identical results. Use the Python API when the merge condition or data come from a DataFrame that isn't easy to express in SQL.

In [0]:
from delta.tables import DeltaTable

In [0]:
# Get reference to the existing Delta table
dt = DeltaTable.forName(spark, f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta")

In [0]:
# ── UPDATE via Python API ─────────────────────────────────────────────────
# Equivalent to: UPDATE customers_delta SET customer_segment='vip' WHERE country='USA'
dt.update(
    condition = F.col("country") == "USA",
    set       = {"customer_segment": F.lit("vip_api")}
)
print("UPDATE via DeltaTable API complete")


display(
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta")
    .filter("country = 'USA' AND customer_segment = 'vip_api'")
    .limit(3)
)

In [0]:
# ── DELETE via Python API ─────────────────────────────────────────────────
# Equivalent to: DELETE FROM customers_delta WHERE customer_segment = 'vip_api'
dt.delete(F.col("customer_segment") == "vip_api")

print("DELETE via DeltaTable API complete")


In [0]:
display(
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta")
    .filter("country = 'USA' AND customer_segment = 'vip_api'")
    .limit(3)
)

In [0]:
# ── MERGE via Python API ──────────────────────────────────────────────────
updates_df = spark.createDataFrame([
    ("CUST010001", "Alice_Updated", "Smith", "alice@new.com"),
    ("CUST999999", "New",           "Person", "new@example.com"),
], ["customer_id", "first_name", "last_name", "email"])


In [0]:
# ── MERGE via Python API ──────────────────────────────────────────────────
(
    dt.alias("target")
    .merge(updates_df.alias("source"), "target.customer_id = source.customer_id")
    .whenMatchedUpdate(set={
        "first_name": "source.first_name",
        "email":      "source.email"
    })
    .whenNotMatchedInsert(values={
        "customer_id": "source.customer_id",
        "first_name":  "source.first_name",
        "last_name":   "source.last_name",
        "email":       "source.email"
    })
    .execute()
)
print("MERGE via DeltaTable Python API complete")
display(
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta")
    .filter("customer_id IN ('CUST010001','CUST999999')")
)

## Change Data Feed (CDF)

Tracking row-level changes (INSERT, UPDATE, DELETE) in Delta tables. CDF enables incremental ETL pipelines by exposing `_change_type` metadata.

**Two terms are often confused in data engineering:**

### Change Data Capture (CDC)
A **pattern/technique** for capturing changes from source systems (databases, APIs, etc.)

| Aspect | Detail |
|--------|--------|
| **Scope** | Source-side technology |
| **What it does** | Captures INSERT, UPDATE, DELETE from operational databases |
| **Tools** | Debezium, AWS DMS, Fivetran, Qlik Replicate |
| **Output** | Stream of change events |

### Change Data Feed (CDF)
A **Delta Lake feature** that records row-level changes within Delta tables.

| Aspect | Detail |
|--------|--------|
| **Scope** | Delta Lake native feature |
| **What it does** | Tracks changes that happen WITHIN Delta tables |
| **Metadata columns** | `_change_type`, `_commit_version`, `_commit_timestamp` |
| **Use case** | Efficient incremental processing — read only changed rows |

> **Pro Tip:** CDC captures changes FROM sources, CDF tracks changes WITHIN Delta tables. They are complementary — CDC feeds data into Delta, CDF enables efficient downstream processing.

**Change Data Feed (CDF) — Syntax Reference**

| Operation | Syntax |
|-----------|--------|
| Enable on existing | `ALTER TABLE t SET TBLPROPERTIES (delta.enableChangeDataFeed = true)` |
| Enable on new table | `CREATE TABLE t (...) TBLPROPERTIES (delta.enableChangeDataFeed = true)` |
| Read by version (SQL) | `SELECT * FROM table_changes('catalog.schema.t', start_version)` |
| Read by timestamp (SQL) | `SELECT * FROM table_changes('t', '2024-01-01 00:00:00')` |
| Read (PySpark) | `spark.read.format("delta").option("readChangeFeed","true").option("startingVersion", 0).table("t")` |
| Stream CDF | `spark.readStream.format("delta").option("readChangeFeed","true").option("startingVersion", 0).table("t")` |
| `_change_type` values | `insert` / `update_preimage` / `update_postimage` / `delete` |

### Example: Enabling Change Data Feed

**Objective:** Enable CDF on a Delta table to track all row-level changes

In [0]:
# Create a table with CDF enabled from the start
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{BRONZE_SCHEMA}.cdf_demo (
    user_id STRING,
    name STRING,
    email STRING,
    status STRING,
    updated_at TIMESTAMP
) 
USING DELTA
TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

print("Table created with Change Data Feed enabled")

In [0]:
# Verify CDF is enabled
properties = spark.sql(f"SHOW TBLPROPERTIES {CATALOG}.{BRONZE_SCHEMA}.cdf_demo")
display(properties.filter(F.col("key").like("%change%")))

### Example: Generating and Tracking Changes

**Objective:** Perform various DML operations and observe how CDF tracks each one

In [0]:
# INSERT initial data
spark.sql(f"""
INSERT INTO {CATALOG}.{BRONZE_SCHEMA}.cdf_demo VALUES
    ('U001', 'Alice', 'alice@example.com', 'active', current_timestamp()),
    ('U002', 'Bob', 'bob@example.com', 'active', current_timestamp()),
    ('U003', 'Charlie', 'charlie@example.com', 'active', current_timestamp())
""")
print("Version 1: Initial INSERT completed")

In [0]:
# UPDATE a record
spark.sql(f"""
UPDATE {CATALOG}.{BRONZE_SCHEMA}.cdf_demo
SET status = 'premium', updated_at = current_timestamp()
WHERE user_id = 'U001'
""")
print("Version 2: UPDATE completed — Alice upgraded to premium")

In [0]:
# DELETE a record
spark.sql(f"""
DELETE FROM {CATALOG}.{BRONZE_SCHEMA}.cdf_demo
WHERE user_id = 'U002'
""")
print("Version 3: DELETE completed — Bob removed")

In [0]:
# INSERT new record
spark.sql(f"""
INSERT INTO {CATALOG}.{BRONZE_SCHEMA}.cdf_demo VALUES
    ('U004', 'Diana', 'diana@example.com', 'trial', current_timestamp())
""")
print("Version 4: INSERT completed — Diana added")

### Example: Reading Change Data Feed

**Objective:** Read and analyze change data with CDF metadata columns

In [0]:
# Current state of the table (standard read)
changes = spark.read \
    .format("delta") \
    .table(f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo")

display(changes)

In [0]:
# Read ALL changes from the beginning using CDF
changes = spark.read \
    .format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 0) \
    .table(f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo")

# Show changes with CDF metadata columns
display(
    changes.select(
        "user_id", "name", "status",
        "_change_type",        # insert, update_preimage, update_postimage, delete
        "_commit_version",     # Delta version number
        "_commit_timestamp"    # When the change occurred
    ).orderBy("_commit_version", "user_id")
)

**Understanding `_change_type` values:**

| Change Type | Description |
|-------------|-------------|
| `insert` | New row inserted |
| `update_preimage` | Row value **BEFORE** update |
| `update_postimage` | Row value **AFTER** update |
| `delete` | Row that was deleted |

> This enables powerful incremental processing patterns — you can process only what changed since the last pipeline run!

In [0]:
# Get only new inserts since version 2
new_inserts = spark.read \
    .format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 2) \
    .table(f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo") \
    .filter(F.col("_change_type") == "insert")

print("New inserts since version 2:")
display(new_inserts.select("user_id", "name", "status", "_commit_version"))

In [0]:
# Get all deletions for audit purposes
deletions = spark.read \
    .format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 0) \
    .table(f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo") \
    .filter(F.col("_change_type") == "delete")

print("All deleted records (for audit):")
display(deletions.select("user_id", "name", "_commit_version", "_commit_timestamp"))

## RESTORE — Disaster Recovery

The `RESTORE` command reverts a Delta table to a previous version — useful for recovering from accidental deletes, bad merges, or schema mistakes.

```sql
-- Restore by version number
RESTORE TABLE my_table TO VERSION AS OF 3;

-- Restore by timestamp
RESTORE TABLE my_table TO TIMESTAMP AS OF '2024-01-15T10:30:00';
```

| Aspect | Details |
|--------|--------|
| **Scope** | Restores data + metadata to exact state at that version |
| **Transaction log** | Creates a NEW commit (does not delete history) |
| **Time travel** | Versions after RESTORE remain accessible |
| **Limitation** | Cannot restore beyond VACUUM retention period |

> **Pro Tip:** RESTORE is a metadata operation — it does not copy data files. It rewrites the Delta log to point at old file versions. This makes it very fast.

**RESTORE & CLONE — Syntax Reference**

| Operation | Syntax |
|-----------|--------|
| Restore to version | `RESTORE TABLE t TO VERSION AS OF 5` |
| Restore to timestamp | `RESTORE TABLE t TO TIMESTAMP AS OF '2024-01-01'` |
| Shallow Clone (instant) | `CREATE TABLE clone_t SHALLOW CLONE source_t` — metadata only, shares data files |
| Deep Clone (full copy) | `CREATE TABLE clone_t DEEP CLONE source_t` — independent copy of all data |
| Clone to version | `CREATE TABLE clone_t DEEP CLONE source_t VERSION AS OF 3` |

In [0]:
# RESTORE demo — revert CDF demo table to version 0
current_version = spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.cdf_demo LIMIT 1").collect()[0]['version']

print(f"Current version: {current_version}")

In [0]:
# Restore to version 1 (after initial insert, before updates)
spark.sql(f"RESTORE TABLE {CATALOG}.{BRONZE_SCHEMA}.cdf_demo TO VERSION AS OF 1")
print(f"Restored to version 1")

In [0]:
# Show the table is back to its original state
restored_count = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo").count()
print(f"Row count after restore: {restored_count}")

In [0]:
restored = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo")
display(restored)

In [0]:
# Check history — RESTORE creates a new version
display(spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.cdf_demo").select('version', 'timestamp', 'operation', 'operationParameters'))

## CLONE — Shallow vs Deep Copy

Delta Lake supports two types of table cloning:

| Aspect | Shallow Clone | Deep Clone |
|--------|--------------|------------|
| **Data files** | References source files (no copy) | Full independent copy |
| **Speed** | Very fast (metadata only) | Slower (copies all data) |
| **Storage cost** | Minimal (shared files) | Full duplicate |
| **Independence** | Depends on source (VACUUM can break it) | Fully independent |
| **Use case** | Testing, short-lived experiments | Migration, backup, environment promotion |

```sql
-- Shallow Clone — fast, references source files
CREATE TABLE my_catalog.dev.customers_test
SHALLOW CLONE my_catalog.prod.customers;

-- Deep Clone — full independent copy
CREATE TABLE my_catalog.backup.customers_backup
DEEP CLONE my_catalog.prod.customers;

-- Clone a specific version (time travel + clone)
CREATE TABLE my_catalog.backup.customers_v5
DEEP CLONE my_catalog.prod.customers VERSION AS OF 5;
```

> **Pro Tip:** Shallow clones are perfect for creating test/dev copies of production tables — zero storage cost, instant creation. But remember: if the source table is VACUUMed, the shallow clone may break. For durable copies, use Deep Clone.

## Shallow Clone

In [0]:
# Create a copy of the cdf_demo table for clone demo
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{BRONZE_SCHEMA}.cdf_demo_for_clone
AS SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.cdf_demo
""")

In [0]:
# --- Shallow Clone Demo ---
# Create a shallow clone of the CDF demo table (instant, no data copy)
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{BRONZE_SCHEMA}.cdf_demo_shallow_clone
SHALLOW CLONE {CATALOG}.{BRONZE_SCHEMA}.cdf_demo_for_clone
""")


In [0]:
# Verify — same data, different table
original_count = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo_for_clone").count()
clone_count = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo_shallow_clone").count()
print(f"Original: {original_count} rows | Shallow Clone: {clone_count} rows")

In [0]:
# Show clone metadata — note "cloneSource" in properties
display(spark.sql(f"DESCRIBE DETAIL {CATALOG}.{BRONZE_SCHEMA}.cdf_demo_shallow_clone"))

In [0]:
# Attempt to insert a row into the shallow clone table
spark.sql(f"""
INSERT INTO {CATALOG}.{BRONZE_SCHEMA}.cdf_demo_shallow_clone
VALUES ('U888', 'ShallowClone User', 'shallow@clone.com', 'test', current_timestamp())
""")

In [0]:
# Select data from the shallow clone table
shallow_clone_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo_shallow_clone")
display(shallow_clone_df)

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.cdf_demo_shallow_clone")

## Deep Clone

In [0]:
# --- Deep Clone Demo ---
# Create a full independent copy (all data files duplicated)
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{BRONZE_SCHEMA}.cdf_demo_deep_clone
DEEP CLONE {CATALOG}.{BRONZE_SCHEMA}.cdf_demo
""")

In [0]:
deep_count = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo_deep_clone").count()
print(f"Deep Clone: {deep_count} rows (fully independent copy)")

In [0]:
# Modify clone without affecting source
spark.sql(f"""
INSERT INTO {CATALOG}.{BRONZE_SCHEMA}.cdf_demo_deep_clone
VALUES ('U999', 'Clone-Only User', 'clone@test.com', 'test', current_timestamp())
""")

In [0]:
# Verify independence
orig = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo").count()
deep = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo_deep_clone").count()
print(f"Original: {orig} rows | Deep Clone: {deep} rows (1 extra row — independent!)")

## Summary

In this notebook, we covered the **fundamentals of Delta Lake**:

| Feature | Purpose | Key Command |
|---------|---------|-------------|
| RESTORE | Disaster recovery | `RESTORE TABLE ... TO VERSION AS OF` |
| **Shallow Clone** | Zero-cost test copies | `CREATE TABLE ... SHALLOW CLONE` |
| **Deep Clone** | Full independent backup | `CREATE TABLE ... DEEP CLONE` |
| VACUUM | Storage cleanup | `VACUUM table RETAIN x HOURS` |
| **Change Data Feed** | **Row-level change tracking** | **`TBLPROPERTIES (delta.enableChangeDataFeed = true)`** |

### Quick Reference — CDF

| Operation | Command |
|-----------|---------|
| Enable CDF | `ALTER TABLE SET TBLPROPERTIES (delta.enableChangeDataFeed = true)` |
| Read CDF | `.option("readChangeFeed", "true").option("startingVersion", N)` |
| CDF metadata | `_change_type`, `_commit_version`, `_commit_timestamp` |

> **Next:** M04 covers Delta optimization techniques (OPTIMIZE, Z-ORDER, Liquid Clustering). CDF for incremental ETL is covered in M05.

## Resource Cleanup

Clean up resources created during the notebook:

In [0]:
# Optional test resource cleanup
# NOTE: Run only if you want to delete all created data

cleanup_tables = [
    "customers_delta",
    "orders_modern",
    "time_travel_demo",
    "cdf_demo",
    "cdf_demo_shallow_clone",
    "cdf_demo_deep_clone"
]

# Uncomment below to execute cleanup:
# for table in cleanup_tables:
#     spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.{table}")
#     print(f"Dropped: {table}")

**[ README](../../../README.md)** | [03 — Optimization & Maintenance](03_optimization_demo.ipynb) →